# Reflex Colab T4 run

This longer runner syncs an exact Git ref, runs the full repository test suite, profiles healthy plus all 11 fault families on a real NVIDIA GPU across multiple seeds, resumes verified runs after interruption, ingests Kineto traces, and publishes the result summary to GitHub.

Set `REFLEX_GITHUB_TOKEN` as a Colab Secret to publish results. Set `REFLEX_REF` to a commit SHA for reproducibility, or leave it unset for the latest `main`. Set `REFLEX_SEEDS` and `REFLEX_ITERS` to increase collection volume.

In [ ]:
# Cell 1 - T4 probe and run directory.
import os, subprocess
from pathlib import Path
probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], text=True, capture_output=True, check=True)
print(probe.stdout.strip())
if 'Tesla T4' not in probe.stdout and 'P100' not in probe.stdout:
    raise RuntimeError('Expected a T4/P100-or-better Colab GPU')
RUNS = Path('/content/reflex_runs')
RUNS.mkdir(exist_ok=True)
print('GPU probe passed')

In [ ]:
# Cell 2 - fetch the latest ref on every session and install test-runner extras.
import os, subprocess, sys
from pathlib import Path
try:
    from google.colab import userdata
    token = userdata.get('REFLEX_GITHUB_TOKEN')
    if token:
        os.environ['REFLEX_GITHUB_TOKEN'] = token
except Exception:
    pass
REPO = Path('/content/reflex')
REMOTE = 'https://github.com/MugiZer/reflex.git'
REFLEX_REF = os.environ.get('REFLEX_REF', 'main')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REMOTE, str(REPO)], check=True)
def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, text=True, capture_output=True).stdout.strip()
git('fetch', 'origin', REFLEX_REF, '--prune')
target = f'origin/{REFLEX_REF}' if REFLEX_REF in ('main', 'reflex-pipeline') else REFLEX_REF
git('checkout', '--detach', target)
COMMIT = git('rev-parse', 'HEAD')
print('testing commit:', COMMIT)
print(git('log', '-1', '--oneline'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest', 'mapie'], check=True)
os.chdir(REPO)


In [ ]:
# Cell 3 - full repository test suite. Keep going so GPU collection still runs on test failures.
import subprocess, sys
tests = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], cwd=REPO, text=True, capture_output=True)
pytest_output = (tests.stdout + '\n' + tests.stderr)[-20000:]
print(pytest_output)
print('full-suite exit code:', tests.returncode)

In [ ]:
# Cell 4 - define the real T4 collection matrix.
import os
from reflex import collect as C
from reflex.collect import nvidia_smi_identity
ident = nvidia_smi_identity()
print('identity:', ident)
if ident['hardware'] == 'unknown':
    raise RuntimeError('nvidia-smi did not provide hardware identity')
faults = ('healthy',) + tuple(C.FAULTS)
seeds = tuple(int(x) for x in os.environ.get('REFLEX_SEEDS', '11,17,23').split(','))
iters = int(os.environ.get('REFLEX_ITERS', '20'))
ROOT = RUNS / COMMIT[:12]
DATASET = ROOT / 'dataset.jsonl'
target_matrix = [(fault, ident['hardware'], C.COLLECTOR_VERSION) for fault in faults]
print({'faults': len(faults), 'seeds': seeds, 'runs': len(faults) * len(seeds), 'iters_per_run': iters, 'root': str(ROOT)})

In [ ]:
# Cell 5 - profile every missing matrix cell with PyTorch Kineto and resume safely.
import json, tempfile
import torch
from torch.profiler import ProfilerActivity, profile
from colab.gpu_workload import run_fault
def device(fault, seed):
    ROOT.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(suffix='.json', dir=ROOT, delete=False) as handle:
        trace_path = Path(handle.name)
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
        stats = run_fault(fault, iters, seed, 'cuda')
    prof.export_chrome_trace(str(trace_path))
    trace = trace_path.read_bytes()
    trace_path.unlink(missing_ok=True)
    return {'trace.json': trace, 'stats.json': json.dumps(stats, sort_keys=True).encode()}
pipeline = C.run_pipeline(
    ROOT, DATASET, target_matrix, faults=faults, seeds=seeds, device=device,
    identity_provider=nvidia_smi_identity, workload='t4-kineto',
    trace_variant='kineto', perf_status='profiled',
    torch_version=torch.__version__, software={'torch': torch.__version__},
)
print(json.dumps(pipeline, indent=2, default=str))

In [ ]:
# Cell 6 - publish tests, collection status, failures, and coverage gaps.
import datetime, json, uuid
from colab.report_results import publish
result = {
    'run_id': datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8],
    'status': 'passed' if tests.returncode == 0 and not pipeline['collected']['failed'] and not pipeline['gaps'] else 'incomplete',
    'commit': COMMIT,
    'requested_ref': REFLEX_REF,
    'hardware': ident,
    'matrix': {'faults': faults, 'seeds': seeds, 'iters_per_run': iters},
    'pytest_exit_code': tests.returncode,
    'pytest_output': pytest_output,
    'pipeline': pipeline,
}
Path('/content/reflex_runs/run_result.json').write_text(json.dumps(result, indent=2, default=str), encoding='utf-8')
publish(result)
print(json.dumps({'status': result['status'], 'records': pipeline['records'], 'gaps': pipeline['gaps']}, indent=2))

In [ ]:
# Cell 7 - optional Drive backup. Copy only after local checksums and ingest complete.
try:
    from google.colab import drive
    import shutil, datetime
    drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/reflex-colab-t4/%s' % datetime.date.today().isoformat()
    shutil.copytree('/content/reflex_runs', dst, dirs_exist_ok=True)
    print('backed up to', dst)
except Exception as exc:
    print('drive backup skipped:', type(exc).__name__, exc)

## Default workload

The default run is 36 profiled cases: one healthy baseline plus the 11 collector fault families, across seeds 11, 17, and 23, with 20 iterations per case. Increase volume with `REFLEX_SEEDS=11,17,23,29,31` and `REFLEX_ITERS=40`.

Results are recorded on the `colab-results` branch; source remains on `main`.